In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
# Rasaan Gouldbourne - AI 240
# Homework 4: Decision Tree Classifier (Loan Repayment Prediction)

import os
import warnings
import numpy as np
import pandas as pd

# Use non-interactive backend so plots save cleanly in Kaggle scripts
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve
)

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 500)
sns.set_theme(style="whitegrid")

# Set Kaggle standard directories
INPUT_DIR = '/kaggle/input'
OUTPUT_DIR = '/kaggle/working'
os.makedirs(OUTPUT_DIR, exist_ok=True)


def step_1_load_data(input_dir: str) -> pd.DataFrame:
    """Step 1: Scan /kaggle/input and load the Loan Repayment CSV dataset."""
    print("=" * 60, flush=True)
    print("STEP 1: SCANNING /kaggle/input FOR LOAN REPAYMENT DATASET", flush=True)
    print("=" * 60, flush=True)

    csv_path = None
    all_found_files = []

    if os.path.exists(input_dir):
        for root, _, files in os.walk(input_dir):
            for file in files:
                full_path = os.path.join(root, file)
                all_found_files.append(full_path)
                # Verify the file is a tabular dataset containing 'result' and 'credit_score'
                if not file.lower().endswith(('.ipynb', '.ipynb.txt')):
                    try:
                        sample_df = pd.read_csv(full_path, nrows=2)
                        cols = [c.lower().strip() for c in sample_df.columns]
                        if 'result' in cols and 'credit_score' in cols:
                            csv_path = full_path
                            break
                    except Exception:
                        continue
            if csv_path:
                break

    print("Files detected in /kaggle/input:", flush=True)
    for f in all_found_files:
        print(f"  -> {f}", flush=True)

    if csv_path and os.path.exists(csv_path):
        print(f"\nLoaded Loan Repayment Dataset: {csv_path}", flush=True)
        df = pd.read_csv(csv_path)
    else:
        raise FileNotFoundError(
            "Could not find the Loan Repayment CSV dataset in /kaggle/input. "
            "Verify that your CSV file containing 'result' and 'credit_score' is attached under Input."
        )

    # Standardize column headers
    df.columns = [col.lower().strip() for col in df.columns]
    return df


def step_2_explore_data(df: pd.DataFrame, output_dir: str) -> None:
    """Step 2 & 3: Explore dataset, identify target variable, and save exploration charts (Rubric: 5 Points)."""
    print("\n" + "=" * 60, flush=True)
    print("STEP 2 & 3: DATA EXPLORATION & TARGET VARIABLE IDENTIFICATION", flush=True)
    print("=" * 60, flush=True)

    print(f"Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns", flush=True)
    print(f"Independent Features: {[c for c in df.columns if c != 'result']}", flush=True)
    print("Target Variable: 'result' ('yes' = Loan Repaid, 'No' = Loan Default/Not Repaid)", flush=True)

    print("\n--- Dataset Info ---", flush=True)
    df.info()

    print("\n--- First 10 Rows ---", flush=True)
    print(df.head(10), flush=True)

    print("\n--- Statistical Summary ---", flush=True)
    print(df.describe().round(2), flush=True)

    # Normalize target text casing temporarily for clean chart labels
    plot_df = df.copy()
    plot_df['result_clean'] = plot_df['result'].astype(str).str.strip().str.capitalize()

    # Exploration Chart 1: Target Class Distribution
    plt.figure(figsize=(6, 4))
    sns.countplot(x='result_clean', hue='result_clean', data=plot_df, palette='Set2', legend=False)
    plt.title('Distribution of Target Variable (Loan Repayment Result)')
    plt.xlabel('Loan Repayment Result (Yes vs. No)')
    plt.ylabel('Applicant Count')
    plt.tight_layout()
    dist_path = os.path.join(output_dir, 'hw4_target_distribution.png')
    plt.savefig(dist_path, dpi=300)
    plt.close()
    print(f"\nSaved Exploration Chart 1: {dist_path}", flush=True)

    # Exploration Chart 2: Feature Correlation Heatmap
    plt.figure(figsize=(8, 6))
    numeric_df = plot_df.select_dtypes(include=[np.number])
    sns.heatmap(numeric_df.corr(), annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
    plt.title('Loan Repayment Features Correlation Matrix')
    plt.tight_layout()
    corr_path = os.path.join(output_dir, 'hw4_correlation_heatmap.png')
    plt.savefig(corr_path, dpi=300)
    plt.close()
    print(f"Saved Exploration Chart 2: {corr_path}", flush=True)

    # Exploration Chart 3: Sum & Credit Score by Loan Repayment Result
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    sns.boxplot(x='result_clean', y='sum', hue='result_clean', data=plot_df, ax=axes[0], palette='Set1', legend=False)
    axes[0].set_title('Loan Sum by Repayment Result')
    axes[0].set_xlabel('Result')
    axes[0].set_ylabel('Sum')

    sns.boxplot(x='result_clean', y='credit_score', hue='result_clean', data=plot_df, ax=axes[1], palette='Set1', legend=False)
    axes[1].set_title('Credit Score by Repayment Result')
    axes[1].set_xlabel('Result')
    axes[1].set_ylabel('Credit Score')

    plt.tight_layout()
    box_path = os.path.join(output_dir, 'hw4_feature_boxplots.png')
    plt.savefig(box_path, dpi=300)
    plt.close()
    print(f"Saved Exploration Chart 3: {box_path}", flush=True)


def step_3_preprocess_data(df: pd.DataFrame):
    """Step 4: Clean data, handle missing values, encode target, and normalize features."""
    print("\n" + "=" * 60, flush=True)
    print("STEP 4: DATA PREPROCESSING & NORMALIZATION", flush=True)
    print("=" * 60, flush=True)

    df_clean = df.copy()

    print("Missing values per column before cleaning:", flush=True)
    print(df_clean.isnull().sum(), flush=True)

    df_clean = df_clean.dropna().drop_duplicates()

    # Standardize target labels ('yes' -> 1, 'no' -> 0)
    df_clean['result'] = df_clean['result'].astype(str).str.strip().str.lower()
    df_clean['target'] = df_clean['result'].map({'yes': 1, 'no': 0})

    if df_clean['target'].isnull().any():
        le = LabelEncoder()
        df_clean['target'] = le.fit_transform(df_clean['result'])

    X = df_clean[['initial_payment', 'last_payment', 'credit_score', 'house_number', 'sum']]
    y = df_clean['target'].astype(int)

    # 80/20 Stratified Train-Test Split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.20, random_state=42, stratify=y
    )

    # Normalize features as requested in the assignment tasks
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    print(f"Clean rows processed: {len(df_clean)}", flush=True)
    print(f"Training set size: {X_train.shape[0]} | Testing set size: {X_test.shape[0]}", flush=True)
    print("Feature normalization (StandardScaler) complete.", flush=True)

    return X_train_scaled, X_test_scaled, y_train, y_test, list(X.columns)


def step_4_train_and_evaluate(
    X_train_scaled: np.ndarray,
    X_test_scaled: np.ndarray,
    y_train: pd.Series,
    y_test: pd.Series,
    feature_names: list,
    output_dir: str
) -> None:
    """Step 5: Train Decision Tree Classifier, evaluate performance, and save visuals (Rubric: 45 Points)."""
    print("\n" + "=" * 60, flush=True)
    print("STEP 5: DECISION TREE MODELING & EVALUATION", flush=True)
    print("=" * 60, flush=True)

    # Initialize and fit Decision Tree Classifier
    dt_model = DecisionTreeClassifier(criterion='entropy', max_depth=4, random_state=42)
    dt_model.fit(X_train_scaled, y_train)
    print("Decision Tree Classifier trained successfully (criterion='entropy', max_depth=4).", flush=True)

    # Generate predictions and probabilities
    y_pred = dt_model.predict(X_test_scaled)
    y_prob = dt_model.predict_proba(X_test_scaled)[:, 1]

    # Calculate evaluation metrics
    acc = accuracy_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_prob)
    report = classification_report(y_test, y_pred, target_names=['No (0)', 'Yes (1)'])
    cm = confusion_matrix(y_test, y_pred)

    print(f"\nTesting Accuracy Score : {acc:.4f} ({acc * 100:.2f}%)", flush=True)
    print(f"ROC-AUC Score          : {roc_auc:.4f}", flush=True)
    print("\nClassification Report:", flush=True)
    print(report, flush=True)
    print("Confusion Matrix:", flush=True)
    print(cm, flush=True)

    # Save Evaluation Report to Kaggle working directory
    report_path = os.path.join(output_dir, 'hw4_decision_tree_evaluation.txt')
    with open(report_path, 'w') as f:
        f.write("Homework 4: Decision Tree Classifier Evaluation Report\n")
        f.write("=" * 55 + "\n")
        f.write(f"Testing Accuracy Score : {acc:.4f} ({acc * 100:.2f}%)\n")
        f.write(f"ROC-AUC Score          : {roc_auc:.4f}\n\n")
        f.write("Classification Report:\n")
        f.write(report + "\n")
        f.write("Confusion Matrix:\n")
        f.write(np.array2string(cm))
    print(f"\nSaved Evaluation Report: {report_path}", flush=True)

    # Plot 1: Confusion Matrix Heatmap
    plt.figure(figsize=(5, 4))
    sns.heatmap(
        cm,
        annot=True,
        fmt='d',
        cmap='Blues',
        cbar=False,
        xticklabels=['No (0)', 'Yes (1)'],
        yticklabels=['No (0)', 'Yes (1)']
    )
    plt.title('Decision Tree Confusion Matrix')
    plt.xlabel('Predicted Loan Repayment')
    plt.ylabel('Actual Loan Repayment')
    plt.tight_layout()
    cm_path = os.path.join(output_dir, 'hw4_confusion_matrix.png')
    plt.savefig(cm_path, dpi=300)
    plt.close()
